In [ ]:
# Célula 1 — Instalação
!pip install osmnx networkx folium matplotlib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 2.8 MB/s eta 0:00:00


In [ ]:
PONTO_A = {"nome": "Museu de Morfologia", "lat": -5.841777929488095, "lon": -35.203041910059596}
PONTO_B = {"nome": "Assai Atacadista",  "lat": -5.858685640917017, "lon": -35.195926594143344}

X = 500  # distância máxima de caminhada em metros

print(f"A: {PONTO_A['nome']}")
print(f"B: {PONTO_B['nome']}")
print(f"Caminhada máxima: {X} m")

A: Museu de Morfologia
B: Assai Atacadista
Caminhada máxima: 500 m


In [ ]:
import osmnx as ox
import networkx as nx

centro = ((PONTO_A["lat"] + PONTO_B["lat"]) / 2,
          (PONTO_A["lon"] + PONTO_B["lon"]) / 2)

# Grafo de CARRO — usado para o trecho P → B
G = ox.graph_from_point(centro, dist=4000, network_type="drive")
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

# Grafo de PEDESTRE — usado apenas para encontrar candidatos P
# e calcular a distância/tempo de caminhada A → P
# Raio menor: só precisa cobrir X metros ao redor de A
G_walk = ox.graph_from_point(
    (PONTO_A["lat"], PONTO_A["lon"]),
    dist=X + 200,           # margem extra de 200m para não cortar candidatos na borda
    network_type="walk"
)

# Nó de A em cada grafo
no_A      = ox.distance.nearest_nodes(G,      PONTO_A["lon"], PONTO_A["lat"])
no_A_walk = ox.distance.nearest_nodes(G_walk, PONTO_A["lon"], PONTO_A["lat"])

# Nó de B apenas no grafo de carro
no_B = ox.distance.nearest_nodes(G, PONTO_B["lon"], PONTO_B["lat"])

print(f"Grafo drive  — Nós: {G.number_of_nodes()} | Arestas: {G.number_of_edges()}")
print(f"Grafo walk   — Nós: {G_walk.number_of_nodes()} | Arestas: {G_walk.number_of_edges()}")
print(f"Nó A (drive): {no_A} | Nó A (walk): {no_A_walk} | Nó B: {no_B}")

Grafo drive  — Nós: 4963 | Arestas: 12356
Grafo walk   — Nós: 925 | Arestas: 2584
Nó A (drive): 501834722 | Nó A (walk): 6956477611 | Nó B: 503296428


In [ ]:
import random
import copy

random.seed(42)

# Candidatos P: exclui o próprio A — P deve ser um ponto DIFERENTE de A
candidatos = nx.single_source_dijkstra_path_length(
    G_walk, no_A_walk, cutoff=X, weight="length"
)
candidatos = {n: d for n, d in candidatos.items() if n != no_A_walk}

print(f"Candidatos a P (≤ {X} m, rede pedestre): {len(candidatos)} nós")

# Trânsito sintético: aplicado apenas ao grafo de CARRO
G_tr = copy.deepcopy(G)
for u, v, data in G_tr.edges(data=True):
    data["travel_time_transito"] = data.get("travel_time", 1.0) * random.uniform(1.0, 3.5)

print("Grafo com trânsito sintético pronto.")

Candidatos a P (≤ 500 m, rede pedestre): 82 nós
Grafo com trânsito sintético pronto.


In [ ]:
import math

def dijkstra_simples(G, origem, destino, peso="length"):
    """Dijkstra sem heap — busca linear pelo menor nó a cada iteração. O(V²)."""
    nos = list(G.nodes)
    dist = {n: math.inf for n in nos}
    ant  = {n: None     for n in nos}
    vis  = set()
    dist[origem] = 0.0

    for _ in range(len(nos)):
        u = min((n for n in nos if n not in vis), key=lambda n: dist[n], default=None)
        if u is None or dist[u] == math.inf or u == destino:
            break
        vis.add(u)

        for v, arestas in G[u].items():
            w = min(a.get(peso, math.inf) for a in arestas.values())
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                ant[v]  = u

    caminho, cur = [], destino
    while cur is not None:
        caminho.append(cur)
        cur = ant[cur]
    caminho.reverse()

    return (dist[destino], caminho) if caminho and caminho[0] == origem else (math.inf, [])

In [ ]:
import heapq

def dijkstra_heap(G, origem, destino, peso="length"):
    """Dijkstra com min-heap. O((V+E) log V)."""
    dist = {origem: 0.0}
    ant  = {origem: None}
    heap = [(0.0, origem)]

    while heap:
        custo, u = heapq.heappop(heap)
        if custo > dist.get(u, math.inf):
            continue
        if u == destino:
            break

        for v, arestas in G[u].items():
            w = min(a.get(peso, math.inf) for a in arestas.values())
            nova = custo + w
            if nova < dist.get(v, math.inf):
                dist[v] = nova
                ant[v]  = u
                heapq.heappush(heap, (nova, v))

    caminho, cur = [], destino
    while cur is not None:
        caminho.append(cur)
        cur = ant.get(cur)
    caminho.reverse()

    return (dist.get(destino, math.inf), caminho) if caminho and caminho[0] == origem else (math.inf, [])

In [ ]:
import math
import random

random.seed(42)

VELOCIDADE_PEDESTRE = 1.4  # m/s

def avaliar_P(G, no_P, no_B, dist_walk, dijkstra_func, peso_tempo="travel_time"):
    """Calcula distância total e tempo total (caminhada + carro) para um dado P."""
    carro_dist,  _ = dijkstra_func(G, no_P, no_B, peso="length")
    carro_tempo, _ = dijkstra_func(G, no_P, no_B, peso=peso_tempo)

    tempo_walk = dist_walk / VELOCIDADE_PEDESTRE

    return {
        "no_P": no_P,
        "walk_m": dist_walk,
        "dist_total_m": dist_walk + carro_dist,
        "tempo_total_s": tempo_walk + carro_tempo,
    }

def melhor_P(G, candidatos, no_B, criterio, dijkstra_func, peso_tempo="travel_time"):
    melhor_custo, melhor_info = math.inf, None
    for no_P, dist_walk in candidatos.items():
        if no_P not in G.nodes:
            continue
        info = avaliar_P(G, no_P, no_B, dist_walk, dijkstra_func, peso_tempo)
        if info[criterio] < melhor_custo:
            melhor_custo = info[criterio]
            melhor_info  = info
    return melhor_info


def cenario4_sem_caminhada(G, G_walk, no_A_walk, no_A_drive, candidatos,
                            no_B, dijkstra_func):

    import osmnx as ox

    # Coordenadas do nó snap de A no grafo drive
    snap_lat = G.nodes[no_A_drive]["y"]
    snap_lon = G.nodes[no_A_drive]["x"]

    # Distância geográfica entre o ponto A real e o nó snap
    dist_snap = ox.distance.great_circle(
        PONTO_A["lat"], PONTO_A["lon"], snap_lat, snap_lon
    )

    TOLERANCIA_M = 1.0  # considera "mesmo ponto" se < 1 metro de diferença

    if dist_snap < TOLERANCIA_M:
        # A exato está no grafo drive — sem caminhada
        no_P       = no_A_drive
        walk_m     = 0.0
        caminhou   = False
    else:
        no_P_mais_proximo = None
        menor_dist        = math.inf
        for no_P_cand, dist_walk_cand in candidatos.items():
            if no_P_cand in G.nodes and dist_walk_cand < menor_dist:
                menor_dist        = dist_walk_cand
                no_P_mais_proximo = no_P_cand

        if no_P_mais_proximo is None:
            # Fallback: usa o nearest_node padrão do osmnx (sem caminhada na rede)
            no_P     = no_A_drive
            walk_m   = dist_snap
            caminhou = True
        else:
            no_P     = no_P_mais_proximo
            walk_m   = menor_dist
            caminhou = True

    carro_dist,  _ = dijkstra_func(G, no_P, no_B, peso="length")
    carro_tempo, _ = dijkstra_func(G, no_P, no_B, peso="travel_time")
    tempo_walk     = walk_m / VELOCIDADE_PEDESTRE

    return {
        "no_P":          no_P,
        "walk_m":        walk_m,
        "dist_total_m":  walk_m + carro_dist,
        "tempo_total_s": tempo_walk + carro_tempo,
        "c4_caminhou":   caminhou,
    }


def calcular_5_cenarios(G, G_tr, G_walk, candidatos, no_A_walk, no_A_drive,
                         no_B, dijkstra_func):
    # Cenários 1, 2, 3 — busca pelo melhor P ≠ A
    c1 = melhor_P(G,    candidatos, no_B, "dist_total_m",  dijkstra_func)
    c2 = melhor_P(G,    candidatos, no_B, "tempo_total_s", dijkstra_func)
    c3 = melhor_P(G_tr, candidatos, no_B, "tempo_total_s", dijkstra_func,
                  peso_tempo="travel_time_transito")

    # Cenário 4 — sem caminhada (parte de A exato ou do P mais próximo)
    c4 = cenario4_sem_caminhada(G, G_walk, no_A_walk, no_A_drive,
                                 candidatos, no_B, dijkstra_func)
    d4_dist  = c4["dist_total_m"]
    d4_tempo = c4["tempo_total_s"]

    # Cenário 5 — ganho ao caminhar (pode ser negativo: significa que não vale a pena)
    ganho_dist  = d4_dist  - c1["dist_total_m"]
    ganho_tempo = d4_tempo - c2["tempo_total_s"]

    # Rótulo do cenário 4 adaptado conforme se houve caminhada forçada
    if c4["c4_caminhou"]:
        label_c4 = f"4 – Sem caminhada (A∉drive, walk {c4['walk_m']:.0f} m)"
    else:
        label_c4 = "4 – Sem caminhada (A→B direto)"

    return [
        ("1 – Menor distância (c/ P)",        c1["dist_total_m"], c1["tempo_total_s"]),
        ("2 – Mais rápida s/ trânsito (c/P)", c2["dist_total_m"], c2["tempo_total_s"]),
        ("3 – Mais rápida c/ trânsito (c/P)", c3["dist_total_m"], c3["tempo_total_s"]),
        (label_c4,                             d4_dist,            d4_tempo),
        ("5 – Ganho ao caminhar (vs cen.4)",  ganho_dist,         ganho_tempo),
    ], (c1, c2, c3, c4)


def imprimir_cenarios(titulo, cenarios):
    print(f"\n=== {titulo} ===")
    print(f"{'Cenário':<52} {'Distância (m)':>14} {'Tempo (s)':>12}")
    print("─" * 82)
    for label, dist, tempo in cenarios:
        d = f"{dist:.1f}"  if isinstance(dist,  float) else dist
        t = f"{tempo:.1f}" if isinstance(tempo, float) else tempo
        print(f"{label:<52} {d:>14} {t:>12}")


# Dijkstra HEAP:
cenarios_heap, (c1h, c2h, c3h, c4h) = calcular_5_cenarios(
    G, G_tr, G_walk, candidatos, no_A_walk, no_A, no_B, dijkstra_heap
)
imprimir_cenarios("Dijkstra Heap (todos os candidatos)", cenarios_heap)

print(f"\nP ótimo (distância)        : nó {c1h['no_P']} — caminhada {c1h['walk_m']:.0f} m"
      f"{' (= A, sem caminhar)' if c1h['walk_m'] == 0 else ''}")
print(f"P ótimo (tempo s/ trânsito) : nó {c2h['no_P']} — caminhada {c2h['walk_m']:.0f} m"
      f"{' (= A, sem caminhar)' if c2h['walk_m'] == 0 else ''}")
print(f"P ótimo (tempo c/ trânsito) : nó {c3h['no_P']} — caminhada {c3h['walk_m']:.0f} m"
      f"{' (= A, sem caminhar)' if c3h['walk_m'] == 0 else ''}")
if c4h["c4_caminhou"]:
    print(f"C4 (sem caminhada)          : A NÃO está no grafo drive → caminhou "
          f"{c4h['walk_m']:.0f} m até nó {c4h['no_P']} (P mais próximo)")
else:
    print(f"C4 (sem caminhada)          : A está no grafo drive → partiu direto do nó {c4h['no_P']}")

# Dijkstra SIMPLES:
cenarios_simples, (c1s, c2s, c3s, c4s) = calcular_5_cenarios(
    G, G_tr, G_walk, candidatos, no_A_walk, no_A, no_B, dijkstra_simples
)
imprimir_cenarios("Dijkstra Simples (amostra de candidatos)", cenarios_simples)



=== Dijkstra Heap (todos os candidatos) ===
Cenário                                               Distância (m)    Tempo (s)
──────────────────────────────────────────────────────────────────────────────────
1 – Menor distância (c/ P)                                   2990.1        533.8
2 – Mais rápida s/ trânsito (c/P)                            3238.5        320.9
3 – Mais rápida c/ trânsito (c/P)                            3238.5        630.9
4 – Sem caminhada (A∉drive, walk 144 m)                      3238.5        320.9
5 – Ganho ao caminhar (vs cen.4)                              248.3          0.0

P ótimo (distância)        : nó 501834700 — caminhada 458 m
P ótimo (tempo s/ trânsito) : nó 500986659 — caminhada 144 m
P ótimo (tempo c/ trânsito) : nó 500986659 — caminhada 144 m
C4 (sem caminhada)          : A NÃO está no grafo drive → caminhou 144 m até nó 500986659 (P mais próximo)

=== Dijkstra Simples (amostra de candidatos) ===
Cenário                                       

In [ ]:
import time
import pandas as pd

linhas = []
for peso, label in [("length", "Distância"), ("travel_time", "Tempo")]:
    t0 = time.perf_counter(); dijkstra_simples(G, no_A, no_B, peso); ts = time.perf_counter() - t0
    t0 = time.perf_counter(); dijkstra_heap(G,   no_A, no_B, peso); th = time.perf_counter() - t0
    linhas.append({"Peso": label, "Simples (s)": ts, "Heap (s)": th, "Speedup": ts/th})

df = pd.DataFrame(linhas).set_index("Peso")
print(df.to_string(float_format="{:.4f}".format))
print(f"\nGrafo: {G.number_of_nodes()} nós | {G.number_of_edges()} arestas")
print("Complexidade — Simples: O(V²) | Heap: O((V+E) log V)")

           Simples (s)  Heap (s)  Speedup
Peso                                     
Distância       0.6651    0.0064 103.6259
Tempo           0.5689    0.0059  97.1008

Grafo: 4963 nós | 12356 arestas
Complexidade — Simples: O(V²) | Heap: O((V+E) log V)


In [ ]:

import folium

def coords_rota(G, caminho):
    """Converte uma lista de nós em lista de coordenadas (lat, lon) para o Folium."""
    return [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in caminho]


def montar_mapa_cenarios(c1, c2, c3, c4, no_A, no_B, titulo_algoritmo):
    """
    Monta um mapa Folium com os 3 cenários (+ C4 sem caminhada) para os
    pontos P escolhidos por um dado algoritmo (c1, c2, c3, c4).

    As rotas A→P e P→B são desenhadas com dijkstra_heap (rápido e exato) —
    o que importa é que o ponto P em si veio do algoritmo comparado
    (heap ou simples).

    C4 agora respeita a nova lógica:
      - Se A estava no grafo drive (c4["c4_caminhou"] == False):
          desenha apenas a rota de carro A→B (sem trecho de caminhada).
      - Se A NÃO estava no grafo drive (c4["c4_caminhou"] == True):
          desenha linha cheia de caminhada A→P4 + linha tracejada de carro P4→B.
    """
    # ── Caminhos de CAMINHADA A→P para cada cenário (rede de pedestre) ─────
    _, rota_walk_c1 = dijkstra_heap(G_walk, no_A_walk, c1["no_P"], peso="length")
    _, rota_walk_c2 = dijkstra_heap(G_walk, no_A_walk, c2["no_P"], peso="length")
    _, rota_walk_c3 = dijkstra_heap(G_walk, no_A_walk, c3["no_P"], peso="length")

    # ── Caminhos de CARRO P→B para cada cenário (rede de carro) ────────────
    _, rota_carro_c1 = dijkstra_heap(G,    c1["no_P"], no_B, peso="length")
    _, rota_carro_c2 = dijkstra_heap(G,    c2["no_P"], no_B, peso="length")
    _, rota_carro_c3 = dijkstra_heap(G_tr, c3["no_P"], no_B, peso="travel_time_transito")

    # ── Cenário 4 — rota depende se A está ou não no grafo drive ────────────
    _, rota_carro_c4 = dijkstra_heap(G, c4["no_P"], no_B, peso="length")
    if c4["c4_caminhou"]:
        # A não estava no grafo drive: há caminhada A→P4
        _, rota_walk_c4 = dijkstra_heap(G_walk, no_A_walk, c4["no_P"], peso="length")
    else:
        rota_walk_c4 = []  # sem caminhada

    mapa = folium.Map(location=[centro[0], centro[1]], zoom_start=14, tiles="CartoDB positron")

    # ── Marcadores de A e B ──────────────────────────────────────────────────
    folium.Marker(
        [PONTO_A["lat"], PONTO_A["lon"]], popup=f"A – {PONTO_A['nome']}",
        icon=folium.Icon(color="green", icon="home")
    ).add_to(mapa)
    folium.Marker(
        [PONTO_B["lat"], PONTO_B["lon"]], popup=f"B – {PONTO_B['nome']}",
        icon=folium.Icon(color="red", icon="flag")
    ).add_to(mapa)

    # ── Marcadores dos pontos de embarque P de cada cenário ─────────────────
    folium.Marker(
        [G.nodes[c1["no_P"]]["y"], G.nodes[c1["no_P"]]["x"]],
        popup=f"P¹ – Menor distância<br>{c1['walk_m']:.0f} m de caminhada",
        icon=folium.Icon(color="blue", icon="car")
    ).add_to(mapa)
    folium.Marker(
        [G.nodes[c2["no_P"]]["y"], G.nodes[c2["no_P"]]["x"]],
        popup=f"P² – Mais rápido s/ trânsito<br>{c2['walk_m']:.0f} m de caminhada",
        icon=folium.Icon(color="purple", icon="car")
    ).add_to(mapa)
    folium.Marker(
        [G.nodes[c3["no_P"]]["y"], G.nodes[c3["no_P"]]["x"]],
        popup=f"P³ – Mais rápido c/ trânsito<br>{c3['walk_m']:.0f} m de caminhada",
        icon=folium.Icon(color="orange", icon="car")
    ).add_to(mapa)

    # Marcador P4 só aparece se houve caminhada forçada
    if c4["c4_caminhou"] and c4["no_P"] != c1["no_P"] and c4["no_P"] != c2["no_P"]:
        folium.Marker(
            [G.nodes[c4["no_P"]]["y"], G.nodes[c4["no_P"]]["x"]],
            popup=f"P⁴ – C4 (P mais próximo)<br>{c4['walk_m']:.0f} m de caminhada",
            icon=folium.Icon(color="gray", icon="car")
        ).add_to(mapa)

    # ── Raio de caminhada máxima a partir de A ───────────────────────────────
    folium.Circle(
        [PONTO_A["lat"], PONTO_A["lon"]], radius=X,
        color="gray", fill=True, fill_opacity=0.07,
        tooltip=f"Raio {X} m"
    ).add_to(mapa)

    # ── Cenário 1 — Menor distância (azul) ───────────────────────────────────
    fg1 = folium.FeatureGroup(name="C1 – Menor distância", show=True)
    folium.PolyLine(coords_rota(G_walk, rota_walk_c1), color="blue", weight=4,
                    tooltip="C1 – Caminhada A→P¹").add_to(fg1)
    folium.PolyLine(coords_rota(G, rota_carro_c1), color="blue", weight=4, dash_array="8",
                    tooltip="C1 – Carro P¹→B").add_to(fg1)
    fg1.add_to(mapa)

    # ── Cenário 2 — Mais rápido s/ trânsito (roxo) ──────────────────────────
    fg2 = folium.FeatureGroup(name="C2 – Mais rápido s/ trânsito", show=True)
    folium.PolyLine(coords_rota(G_walk, rota_walk_c2), color="purple", weight=4,
                    tooltip="C2 – Caminhada A→P²").add_to(fg2)
    folium.PolyLine(coords_rota(G, rota_carro_c2), color="purple", weight=4, dash_array="8",
                    tooltip="C2 – Carro P²→B").add_to(fg2)
    fg2.add_to(mapa)

    # ── Cenário 3 — Mais rápido c/ trânsito (laranja) ───────────────────────
    fg3 = folium.FeatureGroup(name="C3 – Mais rápido c/ trânsito", show=True)
    folium.PolyLine(coords_rota(G_walk, rota_walk_c3), color="orange", weight=4,
                    tooltip="C3 – Caminhada A→P³").add_to(fg3)
    folium.PolyLine(coords_rota(G_tr, rota_carro_c3), color="orange", weight=4, dash_array="8",
                    tooltip="C3 – Carro P³→B (com trânsito)").add_to(fg3)
    fg3.add_to(mapa)

    # ── Cenário 4 — Sem caminhada (cinza, desligado por padrão) ─────────────
    if c4["c4_caminhou"]:
        label_c4   = f"C4 – Sem caminhada (walk forçado {c4['walk_m']:.0f} m)"
        tooltip_walk = f"C4 – Caminhada forçada A→P⁴ ({c4['walk_m']:.0f} m)"
        tooltip_car  = "C4 – Carro P⁴→B"
    else:
        label_c4   = "C4 – Sem caminhada (A→B direto)"
        tooltip_car = "C4 – A→B direto (sem caminhada)"

    fg4 = folium.FeatureGroup(name=label_c4, show=False)
    if c4["c4_caminhou"] and rota_walk_c4:
        folium.PolyLine(coords_rota(G_walk, rota_walk_c4), color="gray", weight=3,
                        opacity=0.7, tooltip=tooltip_walk).add_to(fg4)
    folium.PolyLine(coords_rota(G, rota_carro_c4), color="gray", weight=3, opacity=0.7,
                    dash_array="8", tooltip=tooltip_car).add_to(fg4)
    fg4.add_to(mapa)

    # ── Título flutuante + controle de camadas ──────────────────────────────
    folium.map.Marker(
        [centro[0] + 0.01, centro[1]],
        icon=folium.DivIcon(html=f'<div style="font-size:16px; font-weight:bold; background:white; padding:4px 8px; border-radius:4px;">{titulo_algoritmo}</div>')
    ).add_to(mapa)
    folium.LayerControl(collapsed=False).add_to(mapa)

    return mapa


In [ ]:

# Mapa 1: cenários encontrados pelo Dijkstra HEAP (todos os candidatos)
mapa_heap = montar_mapa_cenarios(c1h, c2h, c3h, c4h, no_A, no_B, "Dijkstra Heap")

# Mapa 2: cenários encontrados pelo Dijkstra SIMPLES (amostra de candidatos)
mapa_simples = montar_mapa_cenarios(c1s, c2s, c3s, c4s, no_A, no_B, "Dijkstra Simples")


In [ ]:
mapa_heap

In [ ]:
mapa_simples